
# Sentinel-1 TOPS InSAR From (Almost) Scratch

This notebook builds an interferogram from two Sentinel-1 **SLC** granules using
as few third-party packages as possible, so that every step of the processing
chain is visible and editable.

**Packages used, and why each is unavoidable:**

| Package | Why we need it | What we *don't* use it for |
|---|---|---|
| `numpy` | array math, FFTs | — |
| `rasterio` (thin wrapper over GDAL) | Sentinel-1 SLC pixels are stored as complex `CInt16` GeoTIFFs; there is no pure-Python reader for these | any InSAR-specific logic |
| `scipy` | `ndimage.uniform_filter` for multilooking, `ndimage.map_coordinates`/FFT for resampling | phase unwrapping (we write that ourselves) |
| `matplotlib` | plotting | — |
| `scikit-image` | **only** used at the very end, as the "known algorithm" we benchmark our hand-written unwrapper against (`skimage.restoration.unwrap_phase`, the Herráez et al. 2002 quality-guided algorithm) | anything else |

Everything else — TOPS deramping, coregistration, interferogram formation,
multilooking, coherence, the Goldstein filter, and phase unwrapping — is
implemented by hand below so you can see exactly what "InSAR processing"
actually does under the hood.

## What this notebook deliberately does *not* do

Full operational InSAR (what ISCE2 / GMTSAR / SNAP do) also includes:

- Precise orbit-based **geometric coregistration** (range-Doppler + DEM), rather
  than the amplitude-cross-correlation coregistration used here.
- **Enhanced Spectral Diversity (ESD)** to fix residual azimuth timing errors
  between bursts (needed for phase accuracy below ~1 mm — not essential for
  learning the concepts).
- **Flat-earth and topographic phase removal** using precise orbits + a DEM.
- **Geocoding** onto a map projection.

These are all straightforward extensions once you're comfortable with the
core pipeline below, and pointers are given at the end. We only process a
**single burst** from a **single subswath**, which is plenty to see fringes
and to learn the algorithm end to end.

> **Validation tip:** because the exact TOPS deramping formula is easy to get
> subtly wrong, the notebook flags every place where you should sanity-check
> your intermediate result (e.g. "the deramped burst should look like a
> stripmap image with no diagonal phase streaking") and, where practical,
> against a trusted open-source implementation (ISCE2's `TopsProc`, or
> GMTSAR's `ESD`) or a SNAP-generated interferogram of the same pair.



## 0. Setup

Edit the paths below. You need two Sentinel-1 IW SLC `.SAFE` folders
(unzipped) covering the same relative orbit / track, over the same area,
with some acceptable temporal baseline (a few days to a few weeks is a good
starting point — long baselines decorrelate).

Folder layout inside a `.SAFE` product that we rely on:

```
S1A_IW_SLC__1SDV_..._....SAFE/
├── annotation/
│   ├── s1a-iw1-slc-vv-....xml      <- per-swath, per-polarization metadata
│   ├── s1a-iw2-slc-vv-....xml
│   └── s1a-iw3-slc-vv-....xml
└── measurement/
    ├── s1a-iw1-slc-vv-....tiff     <- complex pixel data for that swath
    ├── s1a-iw2-slc-vv-....tiff
    └── s1a-iw3-slc-vv-....tiff
```


In [ ]:

import glob, os
import xml.etree.ElementTree as ET
import numpy as np
import rasterio
from scipy.ndimage import uniform_filter, map_coordinates
import matplotlib.pyplot as plt

C = 299792458.0  # speed of light, m/s

# --- EDIT THESE ---
REF_SAFE = "/path/to/S1A_IW_SLC__1SDV_20230101T...SAFE"   # earlier (reference) acquisition
SEC_SAFE = "/path/to/S1A_IW_SLC__1SDV_20230113T...SAFE"   # later (secondary) acquisition
SWATH    = "iw1"   # 'iw1', 'iw2', or 'iw3'
POL      = "vv"    # 'vv' or 'vh' (must exist in both products)
BURST    = 4        # which burst index (0-based) to process - pick one away from the scene edges

def find_annotation(safe_dir, swath, pol):
    pattern = os.path.join(safe_dir, "annotation", f"*-{swath}-slc-{pol}-*.xml")
    hits = glob.glob(pattern)
    assert hits, f"No annotation file found for {pattern}"
    return hits[0]

def find_measurement(safe_dir, swath, pol):
    pattern = os.path.join(safe_dir, "measurement", f"*-{swath}-slc-{pol}-*.tiff")
    hits = glob.glob(pattern)
    assert hits, f"No measurement file found for {pattern}"
    return hits[0]

ref_xml = find_annotation(REF_SAFE, SWATH, POL)
sec_xml = find_annotation(SEC_SAFE, SWATH, POL)
ref_tif = find_measurement(REF_SAFE, SWATH, POL)
sec_tif = find_measurement(SEC_SAFE, SWATH, POL)
print(ref_xml, ref_tif, sec_xml, sec_tif, sep="\n")



## 1. Parse the annotation XML

Everything we need to (a) locate a burst's rows in the TIFF and (b) deramp
its TOPS phase lives in the annotation XML. We parse it with the standard
library `xml.etree.ElementTree` — no extra dependency needed.

Key fields:

- `swathTiming/linesPerBurst`, `samplesPerBurst` — burst dimensions; burst `i`
  occupies rows `[i*linesPerBurst : (i+1)*linesPerBurst]` of the full swath
  TIFF (bursts are stored back-to-back).
- `swathTiming/burstList/burst/azimuthTime` — start time of each burst.
- `imageAnnotation/imageInformation/azimuthTimeInterval` — seconds per azimuth
  line (i.e. 1/PRF-equivalent for the focused image).
- `generalAnnotation/productInformation/azimuthSteeringRate` — TOPS antenna
  electronic steering rate (deg/s), needed for deramping.
- `generalAnnotation/productInformation/radarFrequency` — carrier frequency.
- `generalAnnotation/azimuthFmRateList` — polynomial(s) giving the azimuth FM
  rate `ka` as a function of range.
- `dopplerCentroid/dcEstimateList` — polynomial(s) giving the Doppler
  centroid frequency `fdc` as a function of range.
- `imageAnnotation/imageInformation/slantRangeTime` — two-way slant range
  time to the first range sample (used to build a range-time axis).
- `generalAnnotation/productInformation/rangeSamplingRate` — to convert range
  sample index to range time.
- `generalAnnotation/orbitList` — state vectors, used to get the satellite's
  velocity (for the `ks` term below).


In [ ]:

def text(el, path):
    node = el.find(path)
    return node.text if node is not None else None

def parse_annotation(xml_path):
    root = ET.parse(xml_path).getroot()
    info = {}

    # --- burst geometry ---
    swath_timing = root.find("swathTiming")
    info["lines_per_burst"] = int(text(swath_timing, "linesPerBurst"))
    info["samples_per_burst"] = int(text(swath_timing, "samplesPerBurst"))

    bursts = []
    for b in swath_timing.find("burstList").findall("burst"):
        bursts.append({
            "azimuth_time": np.datetime64(text(b, "azimuthTime")),
            "azimuth_fm_rate_placeholder": None,  # filled in later
            "first_valid_sample": [int(x) for x in text(b, "firstValidSample").split()],
            "last_valid_sample":  [int(x) for x in text(b, "lastValidSample").split()],
        })
    info["bursts"] = bursts

    # --- image / raster geometry ---
    img_info = root.find("imageAnnotation/imageInformation")
    info["azimuth_time_interval"] = float(text(img_info, "azimuthTimeInterval"))  # s / line
    info["slant_range_time"] = float(text(img_info, "slantRangeTime"))            # s, near range two-way
    info["num_samples"] = int(text(img_info, "numberOfSamples"))
    info["num_lines"] = int(text(img_info, "numberOfLines"))

    prod_info = root.find("generalAnnotation/productInformation")
    info["range_sampling_rate"] = float(text(prod_info, "rangeSamplingRate"))  # Hz
    info["radar_frequency"] = float(text(prod_info, "radarFrequency"))         # Hz
    info["azimuth_steering_rate_deg_s"] = float(text(prod_info, "azimuthSteeringRate"))

    # --- azimuth FM rate polynomials (one or more, valid near different times) ---
    fm_list = []
    for fm in root.find("generalAnnotation/azimuthFmRateList").findall("azimuthFmRate"):
        fm_list.append({
            "azimuth_time": np.datetime64(text(fm, "azimuthTime")),
            "t0": float(text(fm, "t0")),
            "coeffs": [float(v) for v in text(fm, "azimuthFmRatePolynomial").split()],
        })
    info["fm_rate_list"] = fm_list

    # --- Doppler centroid polynomials ---
    dc_list = []
    for dc in root.find("dopplerCentroid/dcEstimateList").findall("dcEstimate"):
        dc_list.append({
            "azimuth_time": np.datetime64(text(dc, "azimuthTime")),
            "t0": float(text(dc, "t0")),
            "coeffs": [float(v) for v in text(dc, "dataDcPolynomial").split()],
        })
    info["dc_list"] = dc_list

    # --- orbit state vectors (for satellite velocity) ---
    orbit = []
    for sv in root.find("generalAnnotation/orbitList").findall("orbit"):
        vel = text(sv, "velocity")
        vx, vy, vz = [float(v) for v in [text(sv, "velocity/x"), text(sv, "velocity/y"), text(sv, "velocity/z")]]
        orbit.append({
            "time": np.datetime64(text(sv, "time")),
            "velocity": np.array([vx, vy, vz]),
        })
    info["orbit"] = orbit

    return info

ref_info = parse_annotation(ref_xml)
sec_info = parse_annotation(sec_xml)
print(f"Lines/burst: {ref_info['lines_per_burst']}, samples/burst: {ref_info['samples_per_burst']}")
print(f"Number of bursts: {len(ref_info['bursts'])}")
print(f"Azimuth time interval: {ref_info['azimuth_time_interval']*1000:.4f} ms/line")



## 2. Read one burst's complex pixels

Sentinel-1 SLC pixels are stored as interleaved 16-bit integer (I, Q) pairs,
which GDAL/rasterio exposes directly as a single complex band. We read only
the rows belonging to our chosen burst with a windowed read, so we never have
to load the full ~15,000-line swath into memory.


In [ ]:

def read_burst(tiff_path, info, burst_idx):
    lpb = info["lines_per_burst"]
    row0 = burst_idx * lpb
    with rasterio.open(tiff_path) as src:
        window = rasterio.windows.Window(col_off=0, row_off=row0,
                                          width=src.width, height=lpb)
        data = src.read(1, window=window)  # complex64/complex128 array, shape (lpb, width)
    return data.astype(np.complex64)

ref_burst = read_burst(ref_tif, ref_info, BURST)
sec_burst = read_burst(sec_tif, sec_info, BURST)
print("Reference burst shape:", ref_burst.shape)
print("Secondary burst shape:", sec_burst.shape)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(np.abs(ref_burst), cmap="gray", vmax=np.percentile(np.abs(ref_burst), 98))
axes[0].set_title("Reference amplitude")
axes[1].imshow(np.abs(sec_burst), cmap="gray", vmax=np.percentile(np.abs(sec_burst), 98))
axes[1].set_title("Secondary amplitude")
plt.tight_layout()
plt.show()



## 3. TOPS deramping

Sentinel-1's IW mode electronically steers the antenna beam along-track
during acquisition (Terrain Observation with Progressive Scans). This makes
each point target's Doppler history sweep across a wide bandwidth very
quickly, which shows up as a strong phase ramp/streaking in azimuth that is
*different for every burst* and *different for every range*. Before two
bursts can be compared pixel-by-pixel (coregistered, then interfered), this
synthetic ramp must be removed ("deramping"), and re-added after
interferogram formation if you need the original phase for geocoding.

The steps, following the standard TOPS deramp formulation (see Miranda,
"Definition of the TOPS SLC deramping function for products generated by the
S-1 IPF", ESA, 2015 — cross-check against this reference or an open-source
implementation such as ISCE2's `TopsProc.py` / GMTSAR's `ESD` code if your
result looks wrong):

1. For each **range** sample, evaluate the azimuth FM rate `ka(r)` from the
   nearest-in-time `azimuthFmRateList` polynomial.
2. Compute the Doppler rate induced purely by antenna steering,
   `ks = (2 * v_sat * f_c / c) * steering_rate_rad_per_s`.
3. Compute the combined/"effective" FM rate `kt(r) = ka(r) * ks / (ka(r) - ks)`.
4. For each range, evaluate the Doppler centroid `fdc(r)` from the nearest
   `dcEstimateList` polynomial, and get the reference (beam-center-crossing)
   azimuth time `eta_ref(r) = -fdc(r) / ka(r)`, measured relative to the
   burst's mid-azimuth time.
5. Build the per-pixel deramp phase
   `phi(r, eta) = -pi * kt(r) * (eta - eta_ref(r))**2`
   where `eta` is azimuth time measured from the burst's mid-azimuth-time.
6. Multiply the burst by `exp(-1j * phi)` to deramp it.

**Sanity check:** a correctly deramped burst's amplitude image is unchanged
(deramping only touches phase), but if you look at the *phase* of a bright,
isolated point target before vs after deramping, the wild azimuth phase
sweep should disappear after this step.


In [ ]:

def poly_eval(coeffs, x):
    # Sentinel-1 stores polynomials as [c0, c1, c2, ...] for c0 + c1*x + c2*x^2 + ...
    return sum(c * x**i for i, c in enumerate(coeffs))

def nearest_by_time(items, t):
    return min(items, key=lambda it: abs((it["azimuth_time"] - t) / np.timedelta64(1, "s")))

def satellite_velocity(info, t):
    sv = nearest_by_time(info["orbit"], t)
    return np.linalg.norm(sv["velocity"])

def deramp_phase(info, burst_idx, shape):
    n_lines, n_samples = shape
    burst = info["bursts"][burst_idx]
    t_burst_start = burst["azimuth_time"]
    t_mid = t_burst_start + np.timedelta64(int(n_lines / 2 * info["azimuth_time_interval"] * 1e9), "ns")

    # range time axis (two-way), one value per range sample
    dt_range = 1.0 / info["range_sampling_rate"]
    tau = info["slant_range_time"] + np.arange(n_samples) * dt_range

    fm = nearest_by_time(info["fm_rate_list"], t_burst_start)
    dc = nearest_by_time(info["dc_list"], t_burst_start)

    ka = poly_eval(fm["coeffs"], tau - fm["t0"])           # Hz/s, per range sample
    fdc = poly_eval(dc["coeffs"], tau - dc["t0"])           # Hz, per range sample

    v_sat = satellite_velocity(info, t_mid)
    fc = info["radar_frequency"]
    steering_rate_rad_s = np.deg2rad(info["azimuth_steering_rate_deg_s"])
    ks = (2.0 * v_sat * fc / C) * steering_rate_rad_s        # Hz/s, scalar

    kt = ka * ks / (ka - ks)                                 # Hz/s, per range sample
    eta_ref = -fdc / ka                                      # s, per range sample, relative to t_mid

    # azimuth time of each line, relative to burst mid-time
    eta = (np.arange(n_lines) - n_lines / 2.0) * info["azimuth_time_interval"]

    # broadcast to (n_lines, n_samples)
    phi = -np.pi * kt[None, :] * (eta[:, None] - eta_ref[None, :]) ** 2
    return phi.astype(np.float64)

ref_phi = deramp_phase(ref_info, BURST, ref_burst.shape)
sec_phi = deramp_phase(sec_info, BURST, sec_burst.shape)

ref_deramped = ref_burst * np.exp(-1j * ref_phi)
sec_deramped = sec_burst * np.exp(-1j * sec_phi)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(np.angle(ref_burst[:, ::40]), cmap="twilight", aspect="auto")
axes[0].set_title("Raw phase (subsampled in range)\n-> diagonal TOPS streaking expected")
axes[1].imshow(np.angle(ref_deramped[:, ::40]), cmap="twilight", aspect="auto")
axes[1].set_title("Deramped phase (subsampled in range)\n-> streaking should be gone")
plt.tight_layout()
plt.show()



## 4. Coregistration

Interferometry requires the two SLCs to be aligned to a small fraction of a
pixel. Operationally this is done geometrically (orbits + a DEM), which we
skip here for simplicity. Instead we estimate a single **constant** offset
between the two deramped bursts using amplitude cross-correlation, then
apply a **sub-pixel shift in the frequency domain** (Fourier shift theorem —
multiplying the spectrum by a linear phase ramp is mathematically exact for
band-limited signals, unlike spline interpolation of a complex image, which
can distort phase).

This constant-offset approach is only reasonable because we're comparing
amplitude images of the same scene taken from very similar geometry over a
short segment (one burst); real processors solve for a 2-D polynomial (or
even pixel-by-pixel) offset field, and TOPS mode additionally needs
Enhanced Spectral Diversity to fix residual azimuth timing between bursts.


In [ ]:

def estimate_offset(img1, img2):
    '''Coarse + sub-pixel offset via normalized cross-correlation (phase correlation).'''
    f1 = np.fft.fft2(np.abs(img1) - np.abs(img1).mean())
    f2 = np.fft.fft2(np.abs(img2) - np.abs(img2).mean())
    cross_power = (f1 * np.conj(f2))
    cross_power /= np.abs(cross_power) + 1e-12
    corr = np.fft.ifft2(cross_power)
    corr_mag = np.abs(corr)
    peak = np.unravel_index(np.argmax(corr_mag), corr_mag.shape)
    shifts = np.array(peak)
    # wrap to [-N/2, N/2)
    for i, n in enumerate(corr_mag.shape):
        if shifts[i] > n // 2:
            shifts[i] -= n
    return shifts  # (dy, dx) -- how much img2 must be shifted to match img1

dy, dx = estimate_offset(ref_deramped, sec_deramped)
print(f"Estimated offset (secondary relative to reference): dy={dy} lines, dx={dx} samples")

def shift_fft(img, dy, dx):
    '''Sub-pixel shift of a complex image using the Fourier shift theorem.'''
    ny, nx = img.shape
    fy = np.fft.fftfreq(ny).reshape(-1, 1)
    fx = np.fft.fftfreq(nx).reshape(1, -1)
    ramp = np.exp(-2j * np.pi * (fy * dy + fx * dx))
    return np.fft.ifft2(np.fft.fft2(img) * ramp)

sec_coreg = shift_fft(sec_deramped, dy, dx)

# Trim both images to the overlapping valid region after the integer part of the shift
idy, idx = int(round(dy)), int(round(dx))
y0, y1 = max(0, idy), min(ref_deramped.shape[0], ref_deramped.shape[0] + idy)
x0, x1 = max(0, idx), min(ref_deramped.shape[1], ref_deramped.shape[1] + idx)
ref_common = ref_deramped[y0:y1, x0:x1]
sec_common = sec_coreg[y0:y1, x0:x1]
print("Common overlap shape:", ref_common.shape)



## 5. Form the interferogram, multilook, and estimate coherence

The interferogram is simply `ref * conj(sec)` — its phase is the difference
in range between the two acquisitions (plus flat-earth and topographic
terms, which we don't remove here — see the discussion at the end).

**Multilooking** (spatial averaging) trades resolution for a large reduction
in speckle noise, and lets us estimate **coherence**, the normalized
cross-correlation between the two SLCs in each window — a number from 0
(pure noise) to 1 (perfectly correlated), which tells us where we can trust
the phase.


In [ ]:

interferogram = ref_common * np.conj(sec_common)

# multilooking: choose a window that partially compensates for Sentinel-1's
# ~4:1 azimuth:range pixel spacing ratio, giving roughly square looks on the ground
AZ_LOOKS, RG_LOOKS = 5, 1
window = (AZ_LOOKS, RG_LOOKS)

def multilook(img, window):
    # boxcar average then decimate
    real = uniform_filter(img.real, size=window)
    imag = uniform_filter(img.imag, size=window)
    return (real + 1j * imag)[::window[0], ::window[1]]

def multilook_power(img, window):
    return uniform_filter(np.abs(img) ** 2, size=window)[::window[0], ::window[1]]

ifg_ml = multilook(interferogram, window)
ref_power_ml = multilook_power(ref_common, window)
sec_power_ml = multilook_power(sec_common, window)

coherence = np.abs(ifg_ml) / (np.sqrt(ref_power_ml * sec_power_ml) + 1e-12)
coherence = np.clip(coherence, 0, 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
im0 = axes[0].imshow(np.angle(ifg_ml), cmap="twilight")
axes[0].set_title("Wrapped interferometric phase")
plt.colorbar(im0, ax=axes[0], fraction=0.046)
im1 = axes[1].imshow(coherence, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Coherence")
plt.colorbar(im1, ax=axes[1], fraction=0.046)
im2 = axes[2].imshow(20 * np.log10(np.sqrt(ref_power_ml) + 1e-6), cmap="gray")
axes[2].set_title("Reference amplitude (multilooked, dB)")
plt.tight_layout()
plt.show()

print(f"Mean coherence: {coherence.mean():.3f}")



## 6. Goldstein phase filtering

Even after multilooking, the wrapped phase is noisy. The classic
**Goldstein & Werner (1998)** adaptive filter works patch-by-patch: it takes
the 2-D FFT of a small window of the (complex) interferogram, raises the
magnitude of the spectrum to a power `alpha` (sharpening dominant fringe
frequencies while suppressing noise), and inverts back. Overlapping windows
with a Hann taper avoid blocking artifacts when the patches are stitched
back together.


In [ ]:

def goldstein_filter(cpx, window_size=32, overlap=0.5, alpha=0.6):
    ny, nx = cpx.shape
    step = int(window_size * (1 - overlap))
    win1d = np.hanning(window_size)
    win2d = np.outer(win1d, win1d)

    out = np.zeros_like(cpx)
    weight = np.zeros(cpx.shape, dtype=np.float64)

    for y0 in range(0, ny - window_size + 1, step):
        for x0 in range(0, nx - window_size + 1, step):
            patch = cpx[y0:y0 + window_size, x0:x0 + window_size]
            spec = np.fft.fft2(patch)
            mag = np.abs(spec)
            filt_spec = spec * (mag ** alpha) / (mag.max() + 1e-12)
            filtered_patch = np.fft.ifft2(filt_spec)
            out[y0:y0 + window_size, x0:x0 + window_size] += filtered_patch * win2d
            weight[y0:y0 + window_size, x0:x0 + window_size] += win2d

    weight[weight == 0] = 1.0
    return out / weight

ifg_filtered = goldstein_filter(ifg_ml, window_size=32, overlap=0.5, alpha=0.6)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(np.angle(ifg_ml), cmap="twilight")
axes[0].set_title("Before Goldstein filter")
axes[1].imshow(np.angle(ifg_filtered), cmap="twilight")
axes[1].set_title("After Goldstein filter")
plt.tight_layout()
plt.show()



## 7. Phase unwrapping — hand-written vs. a known algorithm

The interferogram phase is only known modulo 2π ("wrapped"). Unwrapping
recovers the continuous phase surface. We implement a simple
**quality-guided flood-fill unwrapper**: start from the highest-coherence
pixel, and grow outward through a priority queue ordered by coherence,
unwrapping each newly visited pixel relative to its best already-unwrapped
neighbor. This is a simplified version of the algorithm used in many
production tools (a full implementation would also handle residues / branch
cuts explicitly, e.g. Goldstein's branch-cut method or SNAPHU's network-flow
approach).

We then compare our result against **`skimage.restoration.unwrap_phase`**,
which implements the well-established, peer-reviewed Herráez et al. (2002)
quality-guided unwrapping algorithm — a good, easily-installed "known
algorithm" reference. (For a still-stronger benchmark, feed the same wrapped
phase + coherence into SNAPHU or ISCE2/GMTSAR/SNAP and difference the
results the same way.)


In [ ]:

import heapq

def quality_guided_unwrap(phase, quality):
    ny, nx = phase.shape
    unwrapped = np.full((ny, nx), np.nan)
    visited = np.zeros((ny, nx), dtype=bool)

    start = np.unravel_index(np.argmax(quality), quality.shape)
    unwrapped[start] = phase[start]
    visited[start] = True

    heap = []
    def push_neighbors(y, x):
        for ny_, nx_ in ((y-1, x), (y+1, x), (y, x-1), (y, x+1)):
            if 0 <= ny_ < ny and 0 <= nx_ < nx and not visited[ny_, nx_]:
                heapq.heappush(heap, (-quality[ny_, nx_], ny_, nx_, y, x))

    push_neighbors(*start)

    while heap:
        neg_q, y, x, ry, rx = heapq.heappop(heap)
        if visited[y, x]:
            continue
        ref_phase = unwrapped[ry, rx]
        diff = phase[y, x] - ref_phase
        k = np.round(diff / (2 * np.pi))
        unwrapped[y, x] = phase[y, x] - k * 2 * np.pi
        visited[y, x] = True
        push_neighbors(y, x)

    return unwrapped

wrapped_phase = np.angle(ifg_filtered)
my_unwrapped = quality_guided_unwrap(wrapped_phase, coherence)

from skimage.restoration import unwrap_phase
reference_unwrapped = unwrap_phase(wrapped_phase)
# skimage's convention can differ by a global 2*pi*n and/or sign from ours;
# align them on a high-coherence reference pixel before comparing
ref_pixel = np.unravel_index(np.argmax(coherence), coherence.shape)
offset = my_unwrapped[ref_pixel] - reference_unwrapped[ref_pixel]
reference_unwrapped_aligned = reference_unwrapped + offset

diff = my_unwrapped - reference_unwrapped_aligned

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
im0 = axes[0].imshow(my_unwrapped, cmap="jet")
axes[0].set_title("Our quality-guided unwrap")
plt.colorbar(im0, ax=axes[0], fraction=0.046)
im1 = axes[1].imshow(reference_unwrapped_aligned, cmap="jet")
axes[1].set_title("skimage unwrap_phase\n(known algorithm, aligned)")
plt.colorbar(im1, ax=axes[1], fraction=0.046)
im2 = axes[2].imshow(diff, cmap="RdBu", vmin=-2*np.pi, vmax=2*np.pi)
axes[2].set_title("Difference (radians)")
plt.colorbar(im2, ax=axes[2], fraction=0.046)
plt.tight_layout()
plt.show()

print(f"Mean |difference|: {np.nanmean(np.abs(diff)):.4f} rad")
print(f"Std of difference: {np.nanstd(diff):.4f} rad")
print(f"Fraction of pixels differing by > pi (likely unwrapping errors "
      f"in one algorithm or the other): {np.nanmean(np.abs(diff) > np.pi):.4%}")



## 8. Interpreting the comparison

- Large disagreement concentrated in **low-coherence** areas is expected —
  neither algorithm can reliably unwrap noise, and they'll pick different
  (wrong) answers there. Mask by coherence (e.g. `coherence > 0.3`) before
  judging agreement.
- Disagreement in a coherent area that differs by a multiple of exactly
  `2*pi` indicates one algorithm chose a different unwrapping path across a
  residue (a point where phase doesn't integrate to zero around it) —
  inspect where that happens; it's often near strong phase gradients
  (steep terrain, layover, or high deformation gradients if you have real
  ground motion in the scene).
- If you have access to SNAP, ISCE2, or GMTSAR, the strongest test is to
  process the *same* burst/pair through one of those and difference its
  *wrapped* interferogram against `ifg_filtered` here (after accounting for
  the flat-earth phase they remove and we don't) — that validates the whole
  pipeline (deramping + coregistration + interferogram formation), not just
  the unwrapping step.



## 9. Where to go from here

This notebook deliberately stops short of a geocoded, deformation-ready
interferogram. Natural next steps, roughly in order of production-readiness:

1. **Flat-earth phase removal** — using precise orbits and simple geometry
   (or a coarse DEM), subtract the phase due to the reference ellipsoid so
   remaining fringes reflect topography + deformation only.
2. **Topographic phase removal** — subtract a simulated phase from an
   external DEM (e.g. Copernicus GLO-30) to isolate deformation
   ("differential InSAR" / DInSAR).
3. **Multiple bursts / full swath** — stitch adjacent bursts (accounting for
   the ~10% azimuth overlap between them) and adjacent subswaths.
4. **Enhanced Spectral Diversity** — use the burst overlap regions to
   estimate and correct residual azimuth misregistration between the two
   acquisitions, essential for millimeter-level deformation measurements.
5. **Geocoding** — project the unwrapped phase (converted to line-of-sight
   displacement via `displacement = -wavelength / (4*pi) * phase`) onto a
   map grid using the same orbit/DEM geometry.
6. **Compare against a full pipeline** — run the same pair through ISCE2,
   GMTSAR, or ESA SNAP (all free) and compare final products. This is the
   best possible check that nothing in the hand-rolled pipeline above is
   subtly wrong.
